# Streaming with Kafka

Needs a local broker. Easiest options:
```
docker run -d --name kafka -p 9092:9092 apache/kafka:latest
# or the lighter-weight, Kafka-compatible Redpanda:
docker run -d --name redpanda -p 9092:9092 redpandadata/redpanda:latest redpanda start --overprovisioned --smp 1
```
Python client: `pip install confluent-kafka`.

## 1. Connect and create a topic

In [ ]:
from confluent_kafka.admin import AdminClient, NewTopic

BOOTSTRAP = "localhost:9092"
TOPIC = "clicks"

admin = AdminClient({"bootstrap.servers": BOOTSTRAP})
futures = admin.create_topics([NewTopic(TOPIC, num_partitions=3, replication_factor=1)])
for topic, fut in futures.items():
    try:
        fut.result()
        print(f"created topic '{topic}' with 3 partitions")
    except Exception as e:
        print(f"{topic}: {e}")  # fine if it already exists

## 2. Producer: publish JSON click events (keyed by user, so per-user order holds)

In [ ]:
import json
import random
import time

from confluent_kafka import Producer

producer = Producer({"bootstrap.servers": BOOTSTRAP, "acks": "all"})

EVENT_TYPES = ["page_view", "add_to_cart", "purchase"]

def make_event():
    return {"user_id": f"user_{random.randint(1, 20)}",
            "event_type": random.choices(EVENT_TYPES, weights=[8, 3, 1])[0],
            "ts": time.time()}

for _ in range(200):
    event = make_event()
    producer.produce(TOPIC, key=event["user_id"], value=json.dumps(event))
producer.flush()
print("produced 200 events")

## 3. Consumer: read events back

In [ ]:
from confluent_kafka import Consumer

consumer = Consumer({
    "bootstrap.servers": BOOTSTRAP,
    "group.id": "notebook-reader",
    "auto.offset.reset": "earliest",   # first run: start from the beginning
})
consumer.subscribe([TOPIC])

count, samples = 0, []
while count < 200:
    msg = consumer.poll(timeout=5.0)
    if msg is None:
        break
    if msg.error():
        print("error:", msg.error()); continue
    count += 1
    if count <= 5:
        samples.append((msg.partition(), msg.key().decode(), msg.value().decode()))

consumer.close()
print(f"consumed {count} events; first few (partition, key, value):")
for s in samples:
    print(s)
# Note: the same key always lands on the same partition.

## 4. Consumer groups: two consumers split the partitions

In [ ]:
import threading

def run_consumer(name, results):
    c = Consumer({"bootstrap.servers": BOOTSTRAP, "group.id": "shared-group",
                  "auto.offset.reset": "earliest"})
    c.subscribe([TOPIC])
    seen = set()
    end = time.time() + 10
    while time.time() < end:
        msg = c.poll(timeout=1.0)
        if msg and not msg.error():
            seen.add(msg.partition())
    c.close()
    results[name] = seen

results = {}
threads = [threading.Thread(target=run_consumer, args=(f"consumer-{i}", results))
           for i in range(2)]
[t.start() for t in threads]
[t.join() for t in threads]
print(results)
# With 3 partitions and 2 consumers in one group, Kafka splits them e.g. {0,1} / {2}.
# A third consumer would trigger a rebalance; a fourth would sit idle.

## 5. Tumbling-window aggregation: events per type per 10-second window

In [ ]:
from collections import defaultdict

consumer = Consumer({"bootstrap.servers": BOOTSTRAP, "group.id": "windower",
                     "auto.offset.reset": "earliest"})
consumer.subscribe([TOPIC])

WINDOW_SECONDS = 10
windows = defaultdict(lambda: defaultdict(int))  # window_start -> event_type -> count

deadline = time.time() + 8
while time.time() < deadline:
    msg = consumer.poll(timeout=1.0)
    if msg is None or msg.error():
        continue
    event = json.loads(msg.value())
    window_start = int(event["ts"] // WINDOW_SECONDS) * WINDOW_SECONDS
    windows[window_start][event["event_type"]] += 1
consumer.close()

for window_start in sorted(windows):
    print(time.strftime("%H:%M:%S", time.localtime(window_start)), dict(windows[window_start]))

## 6. Mini-project

Run the producer in a terminal loop (continuous stream), keep a windowed consumer running,
and document your delivery-guarantee choices here:
- producer `acks` setting and why
- when your consumer commits offsets (before/after processing) and what a crash replays
- what would make your aggregation idempotent under replay